# Mission 3 — CIMS Analysis (Crosslinking-Induced Mutation Sites)

This notebook identifies protein-RNA binding sites by computing Shannon entropy per position
from CLIP-seq pileup data, then exports bedgraph files for UCSC Genome Browser.

**Genes analysed:** Mirlet7g, Mirlet7f-1, Mirlet7d (mm39)

**Kernel:** `binfo-mission3`

## 1. Imports and paths

In [ ]:
import re
import math
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

WORK    = Path('../data/work')
RESULTS = Path('../results')
RESULTS.mkdir(exist_ok=True)

# Gene definitions: name -> (chrom, start, end, strand)
GENES = {
    'let7g':  ('chr9',  106056039, 106056126, '+'),
    'let7f1': ('chr13',  48691305,  48691393, '-'),
    'let7d':  ('chr13',  48689488,  48689590, '-'),
}

## 2. Helper functions

### 2.1 Clean pileup basereads
Remove structural tags (`^`, `$`, `*`, `<`, `>`, `#`) and insertion/deletion sequences (`+Nxxx`, `-Nxxx`).

In [ ]:
_STRIP = re.compile(r'[<>$*#^]|\+\d+[ACGTNacgtn]+|-\d+[ACGTNacgtn]+')

def clean_basereads(s):
    return _STRIP.sub('', s)

def count_bases(s):
    return Counter(c.upper() for c in s if c.upper() in 'ACGTN.,')

def shannon_entropy(counts):
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return -sum(
        (n / total) * math.log2(n / total)
        for n in counts.values() if n > 0
    )

### 2.2 Process one gene

In [ ]:
def process_gene(name):
    cols = ['chrom', 'pos', '_ref', 'count', 'basereads', 'quals']
    df = pd.read_csv(WORK / f'CLIP-{name}-gene.pileup', sep='\t', names=cols)
    df['matches']     = df['basereads'].apply(clean_basereads)
    df['base_counts'] = df['matches'].apply(count_bases)
    df['entropy']     = df['base_counts'].apply(shannon_entropy)
    df['coverage']    = df['base_counts'].apply(lambda c: sum(c.values()))
    return df

## 3. Load and inspect pileup — Mirlet7g

In [ ]:
df_let7g = process_gene('let7g')
df_let7g[['chrom', 'pos', 'count', 'coverage', 'entropy']].tail(10)

In [ ]:
# Position with maximum entropy in Mirlet7g
idx_max = df_let7g['entropy'].idxmax()
row_max = df_let7g.loc[idx_max]
print(f"Max entropy: {row_max['entropy']:.4f} at position {int(row_max['pos'])}")
print(f"Base counts: {dict(row_max['base_counts'])}")

## 4. Process all three genes

In [ ]:
data = {name: process_gene(name) for name in GENES}

for name, df in data.items():
    print(f"{name}: {len(df)} positions, "
          f"max entropy={df['entropy'].max():.4f}, "
          f"mean coverage={df['coverage'].mean():.1f}")

## 5. Export bedgraph files

In [ ]:
def write_bedgraph(df, name):
    out = RESULTS / f'CLIP-{name}-entropy.bedgraph'
    header = f'track type=bedGraph name="CLIP-{name} Shannon entropy" visibility=full color=0,100,200\n'
    rows = []
    for _, row in df.iterrows():
        s = int(row['pos']) - 1   # 1-based pileup → 0-based bedgraph
        e = int(row['pos'])
        rows.append(f"{row['chrom']}\t{s}\t{e}\t{row['entropy']:.6f}")
    out.write_text(header + '\n'.join(rows) + '\n')
    print(f'Written: {out}')

for name, df in data.items():
    write_bedgraph(df, name)

## 6. Visualisation — Shannon entropy per position

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
labels = {'let7g': 'Mirlet7g (chr9)', 'let7f1': 'Mirlet7f-1 (chr13)', 'let7d': 'Mirlet7d (chr13)'}

for ax, (name, df), color in zip(axes, data.items(), colors):
    ax.bar(df['pos'], df['entropy'], width=1, color=color, alpha=0.8)
    ax.set_title(f'{labels[name]} — CLIP-seq Shannon Entropy')
    ax.set_xlabel('Genomic position')
    ax.set_ylabel('Shannon entropy (bits)')
    ax.set_ylim(0, 2)

    # Mark position of max entropy
    idx_max = df['entropy'].idxmax()
    ax.axvline(df.loc[idx_max, 'pos'], color='red', linestyle='--', linewidth=1,
               label=f"Max at {int(df.loc[idx_max, 'pos'])}")
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(RESULTS / 'entropy_all_genes.pdf', bbox_inches='tight')
fig.savefig(RESULTS / 'entropy_all_genes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figures saved to results/')

## 7. Summary table

In [ ]:
rows = []
for name, (chrom, start, end, strand) in GENES.items():
    df = data[name]
    idx_max = df['entropy'].idxmax()
    rows.append({
        'Gene': f'Mirlet7{name.replace("let7","")}',
        'Region': f'{chrom}:{start}-{end}',
        'Strand': strand,
        'Positions': len(df),
        'Mean coverage': round(df['coverage'].mean(), 1),
        'Max entropy (bits)': round(df['entropy'].max(), 4),
        'Max entropy position': int(df.loc[idx_max, 'pos']),
    })

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / 'entropy_summary.csv', index=False)
summary

## 8. UCSC Genome Browser

Upload the bedgraph files from `results/` to UCSC:

1. Open [UCSC Genome Browser — Mirlet7g (mm39)](http://genome.ucsc.edu/cgi-bin/hgTracks?db=mm39&position=chr9%3A106056039-106056126)
2. Verify genome = **mm39**
3. Click **add custom tracks** → **Choose File** → select `CLIP-let7g-entropy.bedgraph`
4. Submit, then explore the track
5. Export via **View → PDF/PS** and save to `results/`
6. Repeat for `let7f1` and `let7d`